# Jersey City Citibike — Gold Layer Insights

Data basis: Jersey City Citibike, **August 2018 – March 2019** (8 months, ~240,000 trips).

Source: `marts` / `marts_aggregates` schema in Snowflake (Gold Layer).

## Setup — Connecting to Snowflake

In [ ]:
import os
from dotenv import load_dotenv
from cryptography.hazmat.primitives import serialization
import snowflake.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap


load_dotenv("/Users/christian/Documents/next/jersey_city_bikeshare_dbt/.env")

with open(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"], "rb") as key_file:
    p_key = serialization.load_pem_private_key(
        key_file.read(),
        password=None,
    )

pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    private_key=pkb,
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="marts_aggregates"
)

print("Connected to Snowflake successfully.")

## Load Data

Pull all four aggregate/fact sources from the Gold Layer at once, so we don't have to
hit the database again for every section.

In [ ]:
# Hourly pattern by user type (Subscriber vs. Customer)
query_hourly = "SELECT * FROM jersey_city_bikeshare_dev.marts_aggregates.agg_usage_patterns_by_hour"
df = pd.read_sql(query_hourly, conn)

# Daily aggregation (total volume, average duration, etc.)
query_daily = "SELECT * FROM jersey_city_bikeshare_dev.marts_aggregates.agg_daily_summary ORDER BY date_day"
df_daily = pd.read_sql(query_daily, conn)

# Station ranking
query_stations = "SELECT * FROM jersey_city_bikeshare_dev.marts_aggregates.agg_station_ranking ORDER BY total_activity DESC LIMIT 15"
df_stations = pd.read_sql(query_stations, conn)

# Individual trips for distribution analyses (only < 60 min, to cap outliers for the histogram)
query_trips = "SELECT trip_duration_seconds, user_type FROM jersey_city_bikeshare_dev.marts.fct_trips WHERE trip_duration_seconds < 3600"
df_trips = pd.read_sql(query_trips, conn)

# Station coordinates + total activity, for the station map
query_station_coords = """
    SELECT
        s.station_id,
        s.station_name,
        s.station_latitude,
        s.station_longitude,
        COALESCE(r.total_activity, 0) AS total_activity
    FROM jersey_city_bikeshare_dev.marts.dim_stations s
    LEFT JOIN jersey_city_bikeshare_dev.marts_aggregates.agg_station_ranking r
        ON s.station_name = r.station_name
    WHERE s.station_latitude IS NOT NULL AND s.station_longitude IS NOT NULL
"""
df_station_coords = pd.read_sql(query_station_coords, conn)

# Sample of individual trips including start/end coordinates, for the trip route map
query_trip_routes = """
    SELECT
        f.trip_id,
        f.user_type,
        f.trip_duration_seconds,
        s1.station_name AS start_station_name,
        s1.station_latitude AS start_lat,
        s1.station_longitude AS start_lon,
        s2.station_name AS end_station_name,
        s2.station_latitude AS end_lat,
        s2.station_longitude AS end_lon
    FROM jersey_city_bikeshare_dev.marts.fct_trips f
    JOIN jersey_city_bikeshare_dev.marts.dim_stations s1 ON f.start_station_id = s1.station_id
    JOIN jersey_city_bikeshare_dev.marts.dim_stations s2 ON f.end_station_id = s2.station_id
    WHERE f.start_station_id != f.end_station_id
    ORDER BY RANDOM()
    LIMIT 300
"""
df_trip_routes = pd.read_sql(query_trip_routes, conn)

print(f"Hourly pattern: {len(df):,} rows")
print(f"Daily data: {len(df_daily):,} rows ({df_daily['DATE_DAY'].min()} to {df_daily['DATE_DAY'].max()})")
print(f"Stations: {len(df_stations):,} rows")
print(f"Individual trips (<60min): {len(df_trips):,} rows")
print(f"Station coordinates: {len(df_station_coords):,} rows")
print(f"Trip routes (sample): {len(df_trip_routes):,} rows")
df.head()

## Insight 1 — Trip Patterns by Hour & Weekday (Absolute Numbers)

Separate color scales per user type, so the smaller Customer volume isn't washed out
by the much larger Subscriber volume.

In [ ]:
df_sub = df[df["USER_TYPE"] == "Subscriber"]
df_cust = df[df["USER_TYPE"] == "Customer"]

pivot_sub = df_sub.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="TRIP_COUNT", aggfunc="sum", fill_value=0)
pivot_cust = df_cust.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="TRIP_COUNT", aggfunc="sum", fill_value=0)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Subscriber (Commuter)", "Customer (Leisure)"),
    horizontal_spacing=0.15
)

fig.add_trace(
    go.Heatmap(
        z=pivot_sub.values, x=pivot_sub.columns, y=pivot_sub.index,
        colorscale="Viridis", colorbar=dict(title="Trips", x=0.46, len=0.9)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=pivot_cust.values, x=pivot_cust.columns, y=pivot_cust.index,
        colorscale="Plasma", colorbar=dict(title="Trips", x=1.02, len=0.9)
    ),
    row=1, col=2
)

fig.update_layout(
    title="Trip Patterns by Hour & Weekday (Aug 2018 – Mar 2019) — separate scales",
    height=450, width=1100
)
fig.update_xaxes(title_text="Hour of Day", row=1, col=1)
fig.update_xaxes(title_text="Hour of Day", row=1, col=2)
fig.update_yaxes(title_text="Day of Week (0=Sun, 6=Sat)", row=1, col=1)
fig.update_yaxes(title_text="Day of Week (0=Sun, 6=Sat)", row=1, col=2)
fig.show()

## Insight 2 — Relative Usage Pattern (% of Each Group's Own Total Volume)

Both heatmaps normalized to 100% within their own user group, so shape differences
become visible instead of just volume differences.

In [ ]:
df_sub_norm = df_sub.copy()
df_sub_norm["PCT_OF_TYPE"] = df_sub_norm["TRIP_COUNT"] / df_sub_norm["TRIP_COUNT"].sum() * 100

df_cust_norm = df_cust.copy()
df_cust_norm["PCT_OF_TYPE"] = df_cust_norm["TRIP_COUNT"] / df_cust_norm["TRIP_COUNT"].sum() * 100

pivot_sub_norm = df_sub_norm.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="PCT_OF_TYPE", aggfunc="sum", fill_value=0)
pivot_cust_norm = df_cust_norm.pivot_table(index="DAY_OF_WEEK", columns="HOUR_OF_DAY", values="PCT_OF_TYPE", aggfunc="sum", fill_value=0)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Subscriber — Share of Own Volume (%)", "Customer — Share of Own Volume (%)"),
    horizontal_spacing=0.15
)

fig.add_trace(
    go.Heatmap(
        z=pivot_sub_norm.values, x=pivot_sub_norm.columns, y=pivot_sub_norm.index,
        colorscale="Viridis", colorbar=dict(title="% share", x=0.46, len=0.9)
    ),
    row=1, col=1
)

fig.add_trace(
    go.Heatmap(
        z=pivot_cust_norm.values, x=pivot_cust_norm.columns, y=pivot_cust_norm.index,
        colorscale="Plasma", colorbar=dict(title="% share", x=1.02, len=0.9)
    ),
    row=1, col=2
)

fig.update_layout(
    title="Relative Usage Pattern (% of each group's own total volume) — comparable scales",
    height=450, width=1150
)
fig.update_xaxes(title_text="Hour of Day", row=1, col=1)
fig.update_xaxes(title_text="Hour of Day", row=1, col=2)
fig.update_yaxes(title_text="Day of Week (0=Sun, 6=Sat)", row=1, col=1)
fig.update_yaxes(title_text="Day of Week (0=Sun, 6=Sat)", row=1, col=2)
fig.show()

## Insight 3 — Daily Trip Volume Over Time

Now covering the full 8-month period (Aug 2018 – Mar 2019). Seasonal effects
(e.g. a winter cold spell) should be visible here.

In [ ]:
fig = px.line(
    df_daily,
    x="DATE_DAY",
    y="TOTAL_TRIPS",
    title="Daily Trip Volume (Aug 2018 – Mar 2019)",
    labels={"DATE_DAY": "Date", "TOTAL_TRIPS": "Number of Trips"},
    markers=True
)
fig.update_layout(height=420, width=1050)
fig.show()

## Insight 4 — Average Trip Duration Over Time

In [ ]:
fig = px.line(
    df_daily,
    x="DATE_DAY",
    y="AVG_TRIP_DURATION_SECONDS",
    title="Average Trip Duration per Day (Aug 2018 – Mar 2019)",
    labels={"DATE_DAY": "Date", "AVG_TRIP_DURATION_SECONDS": "Avg. Duration (seconds)"},
    markers=True
)
fig.update_layout(height=420, width=1050)
fig.show()

## Insight 5 — Top 15 Stations by Total Activity

Start + end combined, over the full 8-month period.

In [ ]:
fig = px.bar(
    df_stations.sort_values("TOTAL_ACTIVITY"),
    x="TOTAL_ACTIVITY",
    y="STATION_NAME",
    orientation="h",
    title="Top 15 Stations by Total Activity (Start + End), Aug 2018 – Mar 2019",
    labels={"TOTAL_ACTIVITY": "Number of Trips (Start+End)", "STATION_NAME": "Station"}
)
fig.update_layout(height=600, width=1000)
fig.show()

## Insight 5b — Station Map (Locations + Activity)

Each station as a circle: size and color represent total activity (start + end),
over the full 8-month period.

In [ ]:
center_lat = df_station_coords["STATION_LATITUDE"].mean()
center_lon = df_station_coords["STATION_LONGITUDE"].mean()

m_stations = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

max_activity = df_station_coords["TOTAL_ACTIVITY"].max()

for _, row in df_station_coords.iterrows():
    activity_ratio = row["TOTAL_ACTIVITY"] / max_activity if max_activity > 0 else 0
    radius = 4 + activity_ratio * 26  # 4px for inactive stations, up to 30px for the busiest

    folium.CircleMarker(
        location=[row["STATION_LATITUDE"], row["STATION_LONGITUDE"]],
        radius=radius,
        popup=f"{row['STATION_NAME']}<br>{row['TOTAL_ACTIVITY']:,} trips",
        tooltip=row["STATION_NAME"],
        color="#d73027" if activity_ratio > 0.5 else "#4575b4",
        fill=True,
        fill_opacity=0.6,
        weight=1
    ).add_to(m_stations)

m_stations

## Insight 5c — Trip Routes (Sample of Individual Trips)

300 random trips drawn as lines from start to end station (only trips that actually
changed stations). Colored by user type — shows whether Subscribers and Customers use
different travel corridors.

In [ ]:
m_routes = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

color_map = {"Subscriber": "#377eb8", "Customer": "#e41a1c"}

for _, row in df_trip_routes.iterrows():
    folium.PolyLine(
        locations=[[row["START_LAT"], row["START_LON"]], [row["END_LAT"], row["END_LON"]]],
        color=color_map.get(row["USER_TYPE"], "#999999"),
        weight=1.5,
        opacity=0.4,
        tooltip=f"{row['START_STATION_NAME']} -> {row['END_STATION_NAME']} ({row['USER_TYPE']})"
    ).add_to(m_routes)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; background: white;
            padding: 10px; border-radius: 6px; border: 1px solid #999; font-size: 13px;">
  <b>Nutzertyp</b><br>
  <span style="color:#377eb8;">&#9644;&#9644;</span> Subscriber<br>
  <span style="color:#e41a1c;">&#9644;&#9644;</span> Customer
</div>
"""
m_routes.get_root().html.add_child(folium.Element(legend_html))

m_routes

## Insight 5d — Activity Heatmap (All Start Locations)

Instead of individual lines: all trip start points as a density heatmap — shows where
activity is concentrated spatially, independent of fixed station boundaries.

In [ ]:
m_heat = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

heat_data = df_station_coords[["STATION_LATITUDE", "STATION_LONGITUDE", "TOTAL_ACTIVITY"]].values.tolist()

HeatMap(heat_data, radius=25, blur=20, max_zoom=15).add_to(m_heat)

m_heat

## Insight 5e — Cross-System Trips (Jersey City ↔ Manhattan/Brooklyn)

`dim_stations` also contains stations outside Jersey City, because they appear as the
endpoint of some trips. This points to an interconnected Citibike system (a JC bike can
be docked on the other side). Here we quantify how large this share actually is.

Threshold: stations with longitude > -74.02 are considered "across the Hudson"
(Manhattan/Brooklyn side).

In [ ]:
query_cross_system = """
    SELECT
        f.user_type,
        CASE
            WHEN s_end.station_longitude > -74.02 THEN 'Cross-system (ends over there)'
            ELSE 'Jersey City internal'
        END AS trip_type,
        COUNT(*) AS trip_count
    FROM jersey_city_bikeshare_dev.marts.fct_trips f
    JOIN jersey_city_bikeshare_dev.marts.dim_stations s_end ON f.end_station_id = s_end.station_id
    GROUP BY f.user_type, trip_type
"""
df_cross_system = pd.read_sql(query_cross_system, conn)
df_cross_system

In [ ]:
totals_overall = df_cross_system["TRIP_COUNT"].sum()
df_overall = df_cross_system.groupby("TRIP_TYPE")["TRIP_COUNT"].sum().reset_index()

fig = px.pie(
    df_overall,
    names="TRIP_TYPE",
    values="TRIP_COUNT",
    title=f"Share of Cross-System Trips Among All Trips (total: {totals_overall:,})",
    color="TRIP_TYPE",
    color_discrete_map={"Jersey City internal": "#4575b4", "Cross-system (ends over there)": "#d73027"},
    hole=0.4
)
fig.update_traces(textinfo="percent+label")
fig.update_layout(height=420, width=600)
fig.show()

In [ ]:
fig = px.bar(
    df_cross_system,
    x="USER_TYPE",
    y="TRIP_COUNT",
    color="TRIP_TYPE",
    barmode="group",
    title="Cross-System Trips vs. JC-Internal Trips, by User Type",
    labels={"USER_TYPE": "User Type", "TRIP_COUNT": "Number of Trips", "TRIP_TYPE": "Trip Type"},
    color_discrete_map={"Jersey City internal": "#4575b4", "Cross-system (ends over there)": "#d73027"},
    log_y=True
)
fig.update_layout(height=420, width=800)
fig.show()

## Insight 6 — Trip Duration Distribution (Absolute & Normalized)

Absolute counts first (shows actual volume), then normalized to percent (shows the
shape of the distribution independent of group size).

In [ ]:
fig = px.histogram(
    df_trips,
    x="TRIP_DURATION_SECONDS",
    color="USER_TYPE",
    nbins=60,
    title="Trip Duration Distribution (under 60 min) — absolute numbers",
    labels={"TRIP_DURATION_SECONDS": "Trip Duration (seconds)"},
    opacity=0.7,
    barmode="overlay"
)
fig.update_layout(height=420, width=1000)
fig.show()

In [ ]:
fig = px.histogram(
    df_trips,
    x="TRIP_DURATION_SECONDS",
    color="USER_TYPE",
    nbins=60,
    title="Trip Duration Distribution — normalized (% per user type)",
    labels={"TRIP_DURATION_SECONDS": "Trip Duration (seconds)"},
    opacity=0.6,
    barmode="overlay",
    histnorm="percent"
)
fig.update_layout(height=420, width=1000)
fig.show()

## Insight 7 — Trips per Weekday (Absolute & Normalized)

In [ ]:
weekday_names = {0: "Sun", 1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat"}
weekday_order = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

df_weekday = df.groupby(["DAY_OF_WEEK", "USER_TYPE"])["TRIP_COUNT"].sum().reset_index()
df_weekday["WEEKDAY_NAME"] = df_weekday["DAY_OF_WEEK"].map(weekday_names)

fig = px.bar(
    df_weekday,
    x="WEEKDAY_NAME",
    y="TRIP_COUNT",
    color="USER_TYPE",
    barmode="group",
    title="Trips per Weekday, by User Type — absolute numbers",
    labels={"WEEKDAY_NAME": "Weekday", "TRIP_COUNT": "Number of Trips"},
    category_orders={"WEEKDAY_NAME": weekday_order}
)
fig.update_layout(height=420, width=1000)
fig.show()

In [ ]:
totals = df_weekday.groupby("USER_TYPE")["TRIP_COUNT"].sum().reset_index().rename(columns={"TRIP_COUNT": "TOTAL"})
df_weekday_pct = df_weekday.merge(totals, on="USER_TYPE")
df_weekday_pct["PCT"] = df_weekday_pct["TRIP_COUNT"] / df_weekday_pct["TOTAL"] * 100

fig = px.bar(
    df_weekday_pct,
    x="WEEKDAY_NAME",
    y="PCT",
    color="USER_TYPE",
    barmode="group",
    title="Share of Trips per Weekday (% of each group's total volume)",
    labels={"WEEKDAY_NAME": "Weekday", "PCT": "Share (%)"},
    category_orders={"WEEKDAY_NAME": weekday_order}
)
fig.update_layout(height=420, width=1000)
fig.show()

## Summary

| Dimension | Subscriber (Commuter) | Customer (Leisure) |
|---|---|---|
| When | Mon–Fri, 7–9 AM & 5–7 PM | Sat/Sun, 10 AM–3 PM |
| Trip duration | Short, tightly clustered | Longer, widely spread |
| Busiest station | Grove St PATH | — |
| Spatial pattern | See station & route maps (Insight 5b–5d) | See station & route maps |

*Note: figures are based on 8 months of data (Aug 2018 – Mar 2019).*

## Overall Conclusion — Jersey City Citibike Data Analysis (Aug 2018 – Mar 2019)

### Data Basis
- **240,538 trips** over 8 months (August 2018 – March 2019), Jersey City Citibike system
- Processed through a complete medallion architecture (Raw → Staging → Marts) in Snowflake,
  transformed with dbt
- **87 stations**, the large majority within the Jersey City core area
- February/March 2019 (42,171 additional trips) were deliberately withheld and later loaded
  as a simulated "new data arrives" production run through an Airflow/Cosmos pipeline —
  incremental loading worked flawlessly (only the new rows were processed, not the entire
  dataset recomputed)

### Core Finding: Two Clearly Distinguishable User Groups

**Subscribers — ~95% of all trips**
- Classic **commuter behavior**: two sharp activity peaks, morning (7–9 AM) and evening (5–7 PM)
- Concentrated on **weekdays** (Mon–Fri noticeably higher than Sat/Sun)
- Short, tightly clustered trip duration (peak around 250–300 seconds / 4–5 minutes) —
  typical last-mile trips, e.g. to a PATH train station
- Stay almost exclusively within the Jersey City system (only 0.026% of trips end outside it)

**Customers — ~5% of all trips**
- **Leisure/tourist pattern**: strongest activity on Sunday, 10 AM–3 PM, with a secondary,
  smaller hotspot Saturday evening
- Trip duration noticeably longer and more widely spread (a significant share over
  800–2,000+ seconds) — exploratory, less purpose-driven trips
- About 9x more likely to be "cross-system" (ending outside Jersey City) than Subscribers —
  consistent with a more exploratory usage profile

### Spatial Findings
- **Grove St PATH** is by far the most active station (~7,000 trips start+end), followed by
  Exchange Place, Hamilton Park, and Sip Ave — all in close proximity to transit connections,
  further supporting the commuter hypothesis
- The station database also contains a handful of stations in Manhattan/Brooklyn, since the
  Citibike system is technically interconnected there; however, this connection is actually
  used in only **0.037% of all trips** — geographically and operationally, the system is
  de facto confined to Jersey City

### Data Quality
- All 14 automated dbt tests (uniqueness, null checks, referential integrity between fact and
  dimension tables) passed successfully across the full dataset
- During development, the automated tests uncovered and helped resolve a real synchronization
  issue between `fct_trips` and `dim_dates` (a view model wasn't automatically in sync with new
  incremental data)
- The pipeline was tested under a realistic load increase: runtime stayed nearly constant
  (~8 seconds total) when volume increased from 24,910 to 229,554 rows (~9x)

### Business Implications
| Dimension | Subscriber (Commuter) | Customer (Leisure) |
|---|---|---|
| When | Mon–Fri, 7–9 AM & 5–7 PM | Sat/Sun, 10 AM–3 PM |
| Trip duration | Short, tightly clustered (~4–5 min) | Longer, widely spread |
| Weekend share | ~18% of volume | ~41% of volume |
| Cross-system rate | 0.026% | 0.23% |

**Implications for pricing/marketing:** Subscriber offerings should be built around reliable,
short commutes (e.g. annual/monthly passes, focus on station capacity at commuter hubs during
rush hour). Customer offerings should be tailored to weekend/leisure usage (day passes, tourism
marketing, capacity planning at leisure destinations on weekends).

---
*Analysis based on public Citibike trip data (Jersey City), processed through a self-built
dbt/Snowflake pipeline with Airflow orchestration (Cosmos).*